# **Model 3 - Pit Stop Prediction**

Models 1 and 2 were both regression, predicting continuous numbers. This one's different - we're 
predicting whether a lap is a pit lap or not, so it's a classification problem.

Target column is `is_pit_lap`, derived from `PitInTime`. Pit laps are rare compared to normal laps 
(around 3% of the dataset), so class imbalance is going to be a big part of getting this right.

Reusing the same cleanup steps from the last two notebooks (race sessions only, dropping red flag laps, 
deriving is_out_lap). Some features carry over from Models 1 and 2, but not all of them made sense for 
this problem, so only pulling in what's actually relevant to pit timing.

___
## **Initial | Imports | Checks | Target Derivation**

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
df = pd.read_parquet(r"C:\F1-AI\data\processed\fastf1_ml_base.parquet")
print(df.shape)

(469012, 78)


In [3]:
#Race only scope....
df = df[df['SessionName'] == 'Race'].copy()
print(df.shape)

(188482, 78)


In [4]:
# Red flag exclusion - as per anomalies detected while designing model 1.
df = df[df['status_red_flag'] == False].copy()
print(df.shape)

(188056, 78)


In [5]:
df['is_out_lap'] = df['PitOutTime'].notnull()
df['is_out_lap'].value_counts()

is_out_lap
False    182083
True       5973
Name: count, dtype: int64

**Target Derivation**

In [6]:
df['is_pit_lap'] = df['PitInTime'].notnull()
df['is_pit_lap'].value_counts(normalize=True)

is_pit_lap
False    0.969472
True     0.030528
Name: proportion, dtype: float64

___
## **Leakage Features Exclusion | Lagged features**

In [7]:
leakage_cols = [
    'LapTime', 'LapTime_seconds', 'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
    'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
    'Position', 'Deleted', 'DeletedReason',
    'FinalPosition', 'ClassifiedPosition', 'ResultTime', 'Status', 'Points', 'Laps',
    'PitInTime', 'PitOutTime'
]

df_model = df.drop(columns=leakage_cols)
print(df_model.shape)

(188056, 56)


In [8]:
df = df.sort_values(['Season', 'EventName', 'Driver', 'LapNumber'])

grp = df.groupby(['Season', 'EventName', 'Driver'])

df_model['driver_prev_lap_time'] = grp['LapTime_seconds'].shift(1)
df_model['position_prev_lap'] = grp['Position'].shift(1)

print(df_model[['driver_prev_lap_time', 'position_prev_lap']].isnull().sum())

driver_prev_lap_time    5920
position_prev_lap       3411
dtype: int64


In [9]:
n_groups = df_model.groupby(['Season', 'EventName', 'Driver']).ngroups
print(n_groups)

3411


## **Features**

In [10]:
grp_stint = df_model.groupby(['Season', 'EventName', 'Driver', 'Stint'])

df_model['stint_lap_number'] = grp_stint.cumcount() + 1

df_model['tyrelife_sqrt'] = np.sqrt(df_model['TyreLife'])
df_model['tyrelife_log1p'] = np.log1p(df_model['TyreLife'])

df_model['compound_tyrelife'] = df_model['Compound'].astype(str) + '_' + df_model['TyreLife'].astype(str)
df_model['compound_stint_lap'] = df_model['Compound'].astype(str) + '_' + df_model['stint_lap_number'].astype(str)

print(df_model[['stint_lap_number', 'tyrelife_sqrt', 'tyrelife_log1p']].describe())

       stint_lap_number  tyrelife_sqrt  tyrelife_log1p
count      187410.00000  186705.000000   186705.000000
mean           14.46545       3.639679        2.535806
std            10.66622       1.385271        0.757604
min             1.00000       1.000000        0.693147
25%             6.00000       2.645751        2.079442
50%            12.00000       3.605551        2.639057
75%            21.00000       4.582576        3.091042
max            77.00000       8.831761        4.369448


In [11]:
print(df_model['Stint'].isnull().sum())
print(df_model['TyreLife'].isnull().sum())

646
1351


In [12]:
null_stint_rows = df_model[df_model['Stint'].isnull()]
print(null_stint_rows[['Season', 'EventName', 'Driver', 'LapNumber', 'is_out_lap', 'is_pit_lap']].head(15))
print(null_stint_rows['is_out_lap'].value_counts())
print(null_stint_rows['is_pit_lap'].value_counts())

      Season              EventName Driver  LapNumber  is_out_lap  is_pit_lap
4299    2018  Australian_Grand_Prix    GAS        1.0       False       False
4300    2018  Australian_Grand_Prix    RAI        1.0       False       False
4301    2018  Australian_Grand_Prix    SAI        1.0       False       False
4302    2018  Australian_Grand_Prix    VET        1.0       False       False
4303    2018  Australian_Grand_Prix    HAM        1.0       False       False
4304    2018  Australian_Grand_Prix    SIR        1.0       False       False
4305    2018  Australian_Grand_Prix    VER        1.0       False       False
4306    2018  Australian_Grand_Prix    BOT        1.0       False       False
4307    2018  Australian_Grand_Prix    OCO        1.0       False       False
4308    2018  Australian_Grand_Prix    RIC        1.0       False       False
4310    2018  Australian_Grand_Prix    HUL        1.0       False       False
4311    2018  Australian_Grand_Prix    MAG        1.0       Fals

In [13]:
print(null_stint_rows['LapNumber'].value_counts())
print(null_stint_rows.groupby(['Season', 'EventName']).size())

LapNumber
1.0     291
2.0      31
3.0      15
4.0      15
5.0      15
6.0      15
7.0      15
8.0      15
9.0      15
10.0     15
11.0     15
12.0     15
13.0     15
14.0     15
15.0     15
16.0     15
17.0     15
18.0     15
19.0     15
20.0     15
21.0     15
22.0     15
23.0     15
24.0      9
Name: count, dtype: int64
Season  EventName            
2018    Australian_Grand_Prix     20
        Austrian_Grand_Prix       20
        Azerbaijan_Grand_Prix     14
        Bahrain_Grand_Prix        22
        Belgian_Grand_Prix         1
        British_Grand_Prix        21
        Canadian_Grand_Prix       17
        Chinese_Grand_Prix        21
        French_Grand_Prix         14
        German_Grand_Prix         21
        Hungarian_Grand_Prix      20
        Japanese_Grand_Prix       21
        Monaco_Grand_Prix         21
        Russian_Grand_Prix        21
        Singapore_Grand_Prix      20
        Spanish_Grand_Prix        18
2025    Miami_Grand_Prix         354
dtype: int64


In [14]:
non_lap1 = null_stint_rows[null_stint_rows['LapNumber'] != 1]
print(non_lap1.groupby(['Season', 'EventName'])['LapNumber'].agg(['min', 'max', 'count']))

miami = null_stint_rows[null_stint_rows['EventName'] == 'Miami_Grand_Prix']
print(miami['Driver'].value_counts())
print(miami['LapNumber'].min(), miami['LapNumber'].max())

                              min   max  count
Season EventName                              
2018   Australian_Grand_Prix  2.0   2.0      1
       Austrian_Grand_Prix    2.0   2.0      1
       Azerbaijan_Grand_Prix  2.0   2.0      1
       Bahrain_Grand_Prix     2.0   2.0      2
       British_Grand_Prix     2.0   2.0      1
       Canadian_Grand_Prix    2.0   2.0      1
       Chinese_Grand_Prix     2.0   2.0      1
       French_Grand_Prix      2.0   2.0      1
       German_Grand_Prix      2.0   2.0      1
       Hungarian_Grand_Prix   2.0   2.0      1
       Japanese_Grand_Prix    2.0   2.0      1
       Monaco_Grand_Prix      2.0   2.0      1
       Russian_Grand_Prix     2.0   2.0      1
       Singapore_Grand_Prix   2.0   2.0      1
       Spanish_Grand_Prix     2.0   2.0      1
2025   Miami_Grand_Prix       2.0  24.0    339
Driver
VER    24
NOR    24
PIA    24
ALB    24
ANT    24
LEC    24
TSU    24
SAI    24
RUS    24
GAS    23
HUL    23
ALO    23
HAM    23
BEA    23
LAW    

In [15]:
df_model = df_model.sort_values(['Season', 'EventName', 'Driver', 'LapNumber'])

df_model['Stint'] = df_model.groupby(['Season', 'EventName', 'Driver'])['Stint'].ffill()
df_model['Stint'] = df_model['Stint'].fillna(1)

print(df_model['Stint'].isnull().sum())

0


In [16]:
grp_stint = df_model.groupby(['Season', 'EventName', 'Driver', 'Stint'])

df_model['stint_lap_number'] = grp_stint.cumcount() + 1

df_model['tyrelife_sqrt'] = np.sqrt(df_model['TyreLife'])
df_model['tyrelife_log1p'] = np.log1p(df_model['TyreLife'])

df_model['compound_tyrelife'] = df_model['Compound'].astype(str) + '_' + df_model['TyreLife'].astype(str)
df_model['compound_stint_lap'] = df_model['Compound'].astype(str) + '_' + df_model['stint_lap_number'].astype(str)

print(df_model['stint_lap_number'].isnull().sum())
print(df_model[['tyrelife_sqrt', 'tyrelife_log1p']].describe())

0
       tyrelife_sqrt  tyrelife_log1p
count  186705.000000   186705.000000
mean        3.639679        2.535806
std         1.385271        0.757604
min         1.000000        0.693147
25%         2.645751        2.079442
50%         3.605551        2.639057
75%         4.582576        3.091042
max         8.831761        4.369448


#### **SC/VSC Recovery features from model 2**

In [17]:
df_model['is_clean_green_lap'] = (
    df_model['status_green'] &
    ~df_model['status_safety_car'] &
    ~df_model['status_vsc'] &
    ~df_model['status_yellow'] &
    ~df_model['status_red_flag']
)

df_model = df_model.sort_values(['Season', 'EventName', 'LapNumber'])

df_model['is_restart_lap'] = (
    df_model.groupby(['Season', 'EventName'])['is_clean_green_lap']
    .transform(lambda x: x & ~x.shift(1, fill_value=False))
)

print(df_model['is_clean_green_lap'].value_counts())
print(df_model['is_restart_lap'].sum())

is_clean_green_lap
True     166035
False     22021
Name: count, dtype: int64
2284


#### **SC REcovery decay**

In [18]:
df_model['restart_lap_number'] = df_model['LapNumber'].where(df_model['is_restart_lap'])

df_model['restart_lap_number'] = (
    df_model.groupby(['Season', 'EventName'])['restart_lap_number'].ffill()
)

df_model['laps_since_green_resumed'] = df_model['LapNumber'] - df_model['restart_lap_number']
df_model['sc_recovery_decay'] = np.exp(-df_model['laps_since_green_resumed'].fillna(99) / 3)

print(df_model['laps_since_green_resumed'].describe())

count    182170.000000
mean         16.185140
std          15.482417
min           0.000000
25%           4.000000
50%          11.000000
75%          25.000000
max          77.000000
Name: laps_since_green_resumed, dtype: float64


#### **Fuel Load proxy**

In [19]:
max_lap = df_model.groupby(['Season', 'EventName'])['LapNumber'].transform('max')
df_model['fuel_load_pct'] = 1 - (df_model['LapNumber'] - 1) / max_lap
df_model['fuel_load_proxy'] = df_model['fuel_load_pct'] * 110

#### **Race progress**

In [20]:
df_model['race_progress_pct'] = (df_model['LapNumber'] - 1) / max_lap

#### **Commulative green laps**

In [21]:
df_model['cumulative_green_laps'] = (
    df_model.groupby(['Season', 'EventName', 'Driver'])['is_clean_green_lap'].cumsum()
)
df_model['green_laps_sqrt'] = np.sqrt(df_model['cumulative_green_laps'])

#### **Position based features**

In [22]:
df_model['is_race_leader'] = df_model['position_prev_lap'] == 1

df_model['grid_delta'] = df_model['position_prev_lap'] - df_model['GridPosition']

#### **Lagged Weather**

In [23]:
grp = df_model.groupby(['Season', 'EventName', 'Driver'])

for col in ['Rainfall', 'AirTemp', 'TrackTemp', 'Humidity']:
    df_model[f'prev_lap_{col.lower()}'] = grp[col].shift(1)

df_model['track_temp_minus_airtemp'] = df_model['prev_lap_tracktemp'] - df_model['prev_lap_airtemp']

#### **Laps to go and mandatory two compound flag (AI Sugegstion)**

In [24]:
df_model['laps_to_go'] = max_lap - df_model['LapNumber']

compound_count_per_race = (
    df_model.groupby(['Season', 'EventName', 'Driver'])['Compound'].transform('nunique')
)
race_level_multi_compound = (
    df_model.assign(used_multi=compound_count_per_race >= 2)
    .groupby(['Season', 'EventName'])['used_multi'].transform('mean')
)
df_model['is_mandatory_two_compound_race'] = race_level_multi_compound > 0.95

#### **TYpical stint length by compound and track, tyre life pct**

In [25]:
stint_lengths = (
    df_model[df_model['is_pit_lap']]
    .groupby(['EventName', 'Compound'])['TyreLife']
    .quantile(0.75)
    .rename('typical_stint_length')
)

df_model = df_model.merge(stint_lengths, on=['EventName', 'Compound'], how='left')
df_model['tyre_life_pct_of_typical'] = df_model['TyreLife'] / df_model['typical_stint_length']

print(df_model['typical_stint_length'].isnull().sum())

1188


In [26]:
compound_fallback = (
    df_model[df_model['is_pit_lap']]
    .groupby('Compound')['TyreLife']
    .quantile(0.75)
)

df_model['typical_stint_length'] = df_model['typical_stint_length'].fillna(
    df_model['Compound'].map(compound_fallback)
)

df_model['tyre_life_pct_of_typical'] = df_model['TyreLife'] / df_model['typical_stint_length']

print(df_model['typical_stint_length'].isnull().sum())

686


In [27]:
still_null = df_model[df_model['typical_stint_length'].isnull()]
print(still_null['Compound'].value_counts(dropna=False))
print(still_null[['Season', 'EventName']].drop_duplicates())

Compound
nan        646
UNKNOWN     40
Name: count, dtype: int64
        Season              EventName
943       2018  Australian_Grand_Prix
1883      2018    Austrian_Grand_Prix
3128      2018  Azerbaijan_Grand_Prix
3975      2018     Bahrain_Grand_Prix
4991      2018     Belgian_Grand_Prix
6984      2018     British_Grand_Prix
7886      2018    Canadian_Grand_Prix
9105      2018     Chinese_Grand_Prix
10222     2018      French_Grand_Prix
11140     2018      German_Grand_Prix
12392     2018   Hungarian_Grand_Prix
13625     2018    Japanese_Grand_Prix
15847     2018      Monaco_Grand_Prix
17363     2018     Russian_Grand_Prix
18311     2018   Singapore_Grand_Prix
19457     2018     Spanish_Grand_Prix
67660     2021     Belgian_Grand_Prix
178911    2025       Miami_Grand_Prix


umhhh ... XGBoost will handle it , i am leaving this...

#### **Rain increasing and First lap after SC**

In [28]:
df_model['is_rain_increasing'] = df_model['prev_lap_rainfall'] > df_model.groupby(
    ['Season', 'EventName', 'Driver']
)['prev_lap_rainfall'].shift(1)

df_model['is_first_lap_after_sc'] = df_model['is_restart_lap']

#### **Position change and cross-driver leader-pit feature (Very impotant feature i think)**

In [29]:
grp = df_model.groupby(['Season', 'EventName', 'Driver'])
df_model['position_two_laps_ago'] = grp['position_prev_lap'].shift(1)
df_model['position_change_last_lap'] = df_model['position_two_laps_ago'] - df_model['position_prev_lap']

In [30]:
leader_pit_per_lap = (
    df_model[df_model['position_prev_lap'] == 1]
    .groupby(['Season', 'EventName', 'LapNumber'])['is_pit_lap']
    .max()
    .rename('leader_pitted_this_lap')
)

df_model = df_model.merge(
    leader_pit_per_lap, on=['Season', 'EventName', 'LapNumber'], how='left'
)

df_model['leader_pitted_this_lap'] = np.where(
    df_model['leader_pitted_this_lap'].isna(), False, df_model['leader_pitted_this_lap']
).astype(bool)

df_model['leader_pitted_prev_lap'] = (
    df_model.sort_values(['Season', 'EventName', 'LapNumber'])
    .groupby(['Season', 'EventName'])['leader_pitted_this_lap']
    .shift(1)
)

print(df_model['leader_pitted_prev_lap'].value_counts(dropna=False))

leader_pitted_prev_lap
False    179844
True       8040
NaN         172
Name: count, dtype: int64


In [31]:
print(df_model.groupby(['Season', 'EventName']).ngroups)

172


___
## **FEATURE SELECTION AND DROPS**

## Getting the feature list ready

Feature engineering is basically done. Before jumping into modeling, need to sort columns into 
three buckets - stuff that's purely an identifier and should be dropped, categorical stuff that 
needs encoding, and numeric/boolean stuff that's already model-ready.

In [32]:
pd.set_option('display.max_rows', None)
print(df_model.shape)
df_model.dtypes.sort_values()

(188056, 90)


is_pit_lap                                   bool
is_race_leader                               bool
status_vsc                                   bool
status_vsc_ending                            bool
status_unknown                               bool
is_out_lap                                   bool
status_unused3                               bool
status_yellow                                bool
status_green                                 bool
is_mandatory_two_compound_race               bool
IsAccurate                                   bool
status_red_flag                              bool
FastF1Generated                              bool
leader_pitted_this_lap                       bool
is_rain_increasing                           bool
FreshTyre                                    bool
is_first_lap_after_sc                        bool
is_clean_green_lap                           bool
is_restart_lap                               bool
status_safety_car                            bool


In [33]:
drop_cols = [
    'Driver', 'DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId',
    'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'TeamColor', 'TeamId',
    'SessionName',
    'Time', 'LapStartTime', 'LapStartDate', 'Q1', 'Q2', 'Q3',
    'restart_lap_number',
    'AirTemp', 'Humidity', 'TrackTemp', 'Rainfall',
    'TrackStatus',
    'GridPosition'
]

df_model = df_model.drop(columns=drop_cols)
print(df_model.shape)

(188056, 64)


In [34]:
df_model['leader_pitted_prev_lap'] = np.where(
    df_model['leader_pitted_prev_lap'].isna(), False, df_model['leader_pitted_prev_lap']
).astype(bool)

print(df_model['leader_pitted_prev_lap'].value_counts())

leader_pitted_prev_lap
False    180016
True       8040
Name: count, dtype: int64


In [35]:
for col in ['Compound', 'Compound_category', 'Team', 'EventName', 'compound_tyrelife', 'compound_stint_lap']:
    print(col, df_model[col].nunique())

Compound 11
Compound_category 6
Team 19
EventName 36
compound_tyrelife 518
compound_stint_lap 519


In [36]:
df_model = df_model.drop(columns=['compound_tyrelife', 'compound_stint_lap'])
print(df_model.shape)

(188056, 62)


In [37]:
df_model = pd.get_dummies(df_model, columns=['Compound', 'Compound_category', 'Team', 'EventName'], drop_first=False)
print(df_model.shape)

(188056, 130)


___
## **Train/Test Split**


Splitting by race instead of randomly. If laps from the same race end up in both train and test, 
the model could learn race-specific quirks (like exact SC timing) that won't generalize, and the 
validation score would look better than it actually is.

In [38]:
race_keys = df_model[['Season', 'EventName_' + df_model.filter(like='EventName_').columns[0].split('EventName_')[1]]]

In [39]:
print(df_model.columns[df_model.columns.str.contains('EventName') | df_model.columns.str.contains('Season')].tolist()[:5])

['Season', 'EventName_70th_Anniversary_Grand_Prix', 'EventName_Abu_Dhabi_Grand_Prix', 'EventName_Australian_Grand_Prix', 'EventName_Austrian_Grand_Prix']


In [40]:
print(df_model.index.equals(df.index))

False


In [41]:
event_cols = df_model.filter(like='EventName_').columns
df_model['race_id'] = df_model['Season'].astype(str) + '_' + df_model[event_cols].idxmax(axis=1).str.replace('EventName_', '', regex=False)

print(df_model['race_id'].nunique())
print(df_model['race_id'].value_counts().head())

172
race_id
2020_Sakhir_Grand_Prix    1534
2018_Monaco_Grand_Prix    1516
2023_Monaco_Grand_Prix    1515
2019_Monaco_Grand_Prix    1490
2024_Dutch_Grand_Prix     1426
Name: count, dtype: int64


In [42]:
train_mask = df_model['Season'] <= 2024
test_mask = df_model['Season'] == 2025

print(train_mask.sum(), test_mask.sum())
print(df_model.loc[test_mask, 'race_id'].nunique())

161367 26689
24


**Target Drop and define target**

In [43]:
y = df_model['is_pit_lap']

non_feature_cols = ['is_pit_lap', 'race_id', 'Season']
X = df_model.drop(columns=non_feature_cols)

print(X.shape, y.shape)
print(X.dtypes.value_counts())

(188056, 128) (188056,)
bool       92
float64    34
int64       2
Name: count, dtype: int64


**Applying The split on dataframe**


In [44]:
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(161367, 128) (26689, 128)
0.030365564210774198 0.03151110944583911


___
## **Baseline Model Training (XGBoost Baseline)**

In [45]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == False).sum() / (y_train == True).sum()
print(scale_pos_weight)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42
)

model.fit(X_train, y_train)

31.93204081632653


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'aucpr'


In [46]:
from sklearn.metrics import average_precision_score, precision_recall_curve, classification_report

y_proba = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, y_proba)
print("PR-AUC:", pr_auc)

PR-AUC: 0.9290379170569116


In [47]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(20))

IsAccurate                      0.747045
tyre_life_pct_of_typical        0.021299
is_clean_green_lap              0.016632
leader_pitted_this_lap          0.015383
stint_lap_number                0.013886
status_yellow                   0.011701
EventName_French_Grand_Prix     0.006382
status_safety_car               0.005809
status_unknown                  0.005118
position_prev_lap               0.004827
is_out_lap                      0.003920
EventName_Styrian_Grand_Prix    0.003788
avg_throttle                    0.003761
laps_since_green_resumed        0.003348
status_vsc                      0.003318
avg_speed                       0.003292
race_progress_pct               0.003057
status_vsc_ending               0.002959
typical_stint_length            0.002953
FastF1Generated                 0.002743
dtype: float32


**Highly Suspicious - probably feature leaked , IsAccurate in FastF1 is a data- quality flag , which is False when the lap is affected by the things like pit in / pit out or safety car interference / or timing analysis**

In [48]:
print(pd.crosstab(df_model['IsAccurate'], df_model['is_pit_lap'], normalize='index'))

is_pit_lap     False     True 
IsAccurate                    
False       0.781902  0.218098
True        1.000000  0.000000


### **Re-Training after dropping IsAccurate feature**

In [49]:
X_train = X_train.drop(columns=['IsAccurate'])
X_test = X_test.drop(columns=['IsAccurate'])

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42
)

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, y_proba)
print("PR-AUC:", pr_auc)

PR-AUC: 0.6703467734234931


Ohh, now it reduced to expected baseline number , we should see feature importancve now....

**Feature Importance**

In [50]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(20))

race_progress_pct                     0.063320
leader_pitted_this_lap                0.039480
tyre_life_pct_of_typical              0.035406
fuel_load_pct                         0.027333
EventName_Monaco_Grand_Prix           0.027301
status_yellow                         0.026640
EventName_French_Grand_Prix           0.022490
EventName_Hungarian_Grand_Prix        0.022032
EventName_Italian_Grand_Prix          0.020517
leader_pitted_prev_lap                0.019933
EventName_São_Paulo_Grand_Prix        0.019457
avg_speed                             0.018207
EventName_Azerbaijan_Grand_Prix       0.017977
EventName_Singapore_Grand_Prix        0.017791
EventName_United_States_Grand_Prix    0.017347
typical_stint_length                  0.016908
EventName_Chinese_Grand_Prix          0.015030
LapNumber                             0.014745
position_prev_lap                     0.014711
EventName_Belgian_Grand_Prix          0.014329
dtype: float32


In [51]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = f1_scores.argmax()

print("Best threshold:", thresholds[best_idx])
print("Precision:", precisions[best_idx])
print("Recall:", recalls[best_idx])
print("F1:", f1_scores[best_idx])

Best threshold: 0.8278828
Precision: 0.7261724659606656
Recall: 0.5707491082045184
F1: 0.6391478024366082


**Threshold table**

In [52]:
sweep_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for t in sweep_thresholds:
    preds = (y_proba >= t).astype(int)
    p = precisions[(thresholds >= t).argmax()] if t <= thresholds.max() else precisions[-1]
    
print("threshold  precision  recall  f1")
for t in sweep_thresholds:
    idx = (thresholds >= t).argmax()
    p, r = precisions[idx], recalls[idx]
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f"{t:.1f}        {p:.3f}      {r:.3f}   {f1:.3f}")

threshold  precision  recall  f1
0.2        0.153      0.925   0.263
0.3        0.203      0.901   0.331
0.4        0.253      0.859   0.391
0.5        0.323      0.806   0.462
0.6        0.408      0.754   0.530
0.7        0.511      0.672   0.581
0.8        0.676      0.597   0.634
0.9        0.842      0.411   0.553


## **Adding one important feature**
___

In [54]:
pit_by_position = (
    df_model.groupby(['race_id', 'LapNumber', 'position_prev_lap'])['is_pit_lap']
    .max()
    .reset_index()
    .rename(columns={'is_pit_lap': 'car_ahead_pitted_this_lap', 'position_prev_lap': 'lookup_position'})
)

df_model['lookup_position'] = df_model['position_prev_lap'] - 1

df_model = df_model.merge(
    pit_by_position,
    left_on=['race_id', 'LapNumber', 'lookup_position'],
    right_on=['race_id', 'LapNumber', 'lookup_position'],
    how='left'
)

df_model['car_ahead_pitted_this_lap'] = np.where(
    df_model['car_ahead_pitted_this_lap'].isna(), False, df_model['car_ahead_pitted_this_lap']
).astype(bool)

df_model['car_ahead_pitted_prev_lap'] = (
    df_model.sort_values(['race_id', 'LapNumber'])
    .groupby('race_id')['car_ahead_pitted_this_lap']
    .shift(1)
)

df_model['car_ahead_pitted_prev_lap'] = np.where(
    df_model['car_ahead_pitted_prev_lap'].isna(), False, df_model['car_ahead_pitted_prev_lap']
).astype(bool)

df_model = df_model.drop(columns=['lookup_position'])

print(df_model['car_ahead_pitted_prev_lap'].value_counts())

car_ahead_pitted_prev_lap
False    182719
True       5337
Name: count, dtype: int64


In [57]:
df_model = df_model.drop(columns=['IsAccurate'])

X = df_model.drop(columns=['is_pit_lap', 'race_id', 'Season'])
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(X_train.shape, X_test.shape)
print('IsAccurate' in X_train.columns)

(161367, 129) (26689, 129)
False


In [58]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42
)

model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, y_proba)
print("PR-AUC:", pr_auc)

PR-AUC: 0.6784169667620545


In [59]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(20))

race_progress_pct                  0.059766
car_ahead_pitted_prev_lap          0.037395
status_yellow                      0.036295
leader_pitted_this_lap             0.033683
tyre_life_pct_of_typical           0.030875
car_ahead_pitted_this_lap          0.030508
fuel_load_pct                      0.030452
EventName_Italian_Grand_Prix       0.024658
EventName_Monaco_Grand_Prix        0.022160
EventName_São_Paulo_Grand_Prix     0.020089
EventName_French_Grand_Prix        0.019962
EventName_Hungarian_Grand_Prix     0.018445
EventName_Azerbaijan_Grand_Prix    0.018251
avg_speed                          0.017314
leader_pitted_prev_lap             0.016888
EventName_Singapore_Grand_Prix     0.016068
EventName_Belgian_Grand_Prix       0.014740
position_prev_lap                  0.014679
Compound_MEDIUM                    0.014506
typical_stint_length               0.014158
dtype: float32


In [60]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = f1_scores.argmax()

print("Best threshold:", thresholds[best_idx])
print("Precision:", precisions[best_idx])
print("Recall:", recalls[best_idx])
print("F1:", f1_scores[best_idx])

Best threshold: 0.7929103
Precision: 0.6773333333333333
Recall: 0.6040428061831153
F1: 0.6385920799541812


### **Hyperparameter Tuning (Manual)**

In [62]:
train_races = df_model.loc[train_mask, 'race_id'].unique()
np.random.seed(42)
val_races = np.random.choice(train_races, size=int(len(train_races) * 0.15), replace=False)

val_mask = df_model['race_id'].isin(val_races) & train_mask
train2_mask = train_mask & ~val_mask

X_train2, X_val = X[train2_mask], X[val_mask]
y_train2, y_val = y[train2_mask], y[val_mask]

print(X_train2.shape, X_val.shape)

(136372, 129) (24995, 129)


In [63]:
import itertools
import time

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [200, 300, 400],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

all_combos = list(itertools.product(*param_grid.values()))
np.random.seed(42)
sampled_combos = [all_combos[i] for i in np.random.choice(len(all_combos), size=25, replace=False)]

param_names = list(param_grid.keys())
results = []

for combo in tqdm(sampled_combos, desc="tuning"):
    params = dict(zip(param_names, combo))
    m = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr',
        random_state=42,
        n_jobs=-1
    )
    m.fit(X_train2, y_train2, verbose=False)
    proba = m.predict_proba(X_val)[:, 1]
    score = average_precision_score(y_val, proba)
    results.append({**params, 'pr_auc': score})

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
results_df.head(10)

tuning: 100%|██████████████████████████████████████████████████████████████████████████| 25/25 [04:01<00:00,  9.66s/it]


,max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,pr_auc
15,8,0.10,300,1,0.9,0.7,0.716129
20,8,0.10,400,1,0.7,0.9,0.710823
7,6,0.10,300,5,0.7,0.7,0.690161
16,8,0.10,200,10,0.7,0.7,0.685385
21,8,0.05,300,10,0.7,0.7,0.680147
4,6,0.10,200,1,0.7,0.7,0.672673
10,6,0.05,400,10,0.9,0.9,0.668494
8,8,0.05,300,1,0.9,0.7,0.665236
14,6,0.10,200,5,0.9,0.7,0.658556
11,6,0.05,300,10,0.7,0.7,0.632878


In [64]:
best_params = results_df.iloc[0][param_names].to_dict()
best_params['max_depth'] = int(best_params['max_depth'])
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['min_child_weight'] = int(best_params['min_child_weight'])

print(best_params)

final_model = XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

y_proba_final = final_model.predict_proba(X_test)[:, 1]
pr_auc_final = average_precision_score(y_test, y_proba_final)
print("Final PR-AUC on 2025 holdout:", pr_auc_final)

{'max_depth': 8, 'learning_rate': 0.1, 'n_estimators': 300, 'min_child_weight': 1, 'subsample': 0.9, 'colsample_bytree': 0.7}
Final PR-AUC on 2025 holdout: 0.7004412346747299


In [65]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_final)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = f1_scores.argmax()

print("Best threshold:", thresholds[best_idx])
print("Precision:", precisions[best_idx])
print("Recall:", recalls[best_idx])
print("F1:", f1_scores[best_idx])

Best threshold: 0.68261945
Precision: 0.7839195979899497
Recall: 0.5564803804994055
F1: 0.6509040328940897


In [66]:
sweep_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print("threshold  precision  recall  f1")
for t in sweep_thresholds:
    idx = (thresholds >= t).argmax()
    p, r = precisions[idx], recalls[idx]
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f"{t:.1f}        {p:.3f}      {r:.3f}   {f1:.3f}")

threshold  precision  recall  f1
0.2        0.381      0.757   0.507
0.3        0.472      0.703   0.565
0.4        0.561      0.663   0.608
0.5        0.653      0.623   0.637
0.6        0.718      0.587   0.646
0.7        0.795      0.546   0.647
0.8        0.864      0.498   0.632
0.9        0.921      0.427   0.583


___
### **Adding some Opponent driver based features to boost signal**

**Raw compound and Team Reconstruct**

In [69]:
compound_cols = [c for c in df_model.columns if c.startswith('Compound_') and not c.startswith('Compound_category_')]
team_cols = [c for c in df_model.columns if c.startswith('Team_')]

df_model['raw_compound'] = df_model[compound_cols].idxmax(axis=1).str.replace('Compound_', '', regex=False)
df_model['raw_team'] = df_model[team_cols].idxmax(axis=1).str.replace('Team_', '', regex=False)

print(df_model['raw_compound'].value_counts())
print(df_model['raw_team'].nunique())

raw_compound
HARD            68633
MEDIUM          64358
SOFT            33143
INTERMEDIATE     9292
SUPERSOFT        5549
ULTRASOFT        4323
HYPERSOFT         887
WET               726
nan               646
None              459
UNKNOWN            40
Name: count, dtype: int64
19


**Leader/Car ahead Compound and tyrelife**

In [70]:
position_state = df_model[['race_id', 'LapNumber', 'position_prev_lap', 'raw_compound', 'TyreLife']]

leader_state = position_state[position_state['position_prev_lap'] == 1][['race_id', 'LapNumber', 'raw_compound', 'TyreLife']]
leader_state = leader_state.rename(columns={'raw_compound': 'leader_compound', 'TyreLife': 'leader_tyrelife'})

df_model = df_model.merge(leader_state, on=['race_id', 'LapNumber'], how='left')

df_model['lookup_position'] = df_model['position_prev_lap'] - 1

car_ahead_state = position_state.rename(columns={'position_prev_lap': 'lookup_position', 'raw_compound': 'car_ahead_compound', 'TyreLife': 'car_ahead_tyrelife'})

df_model = df_model.merge(car_ahead_state, on=['race_id', 'LapNumber', 'lookup_position'], how='left')

df_model = df_model.drop(columns=['lookup_position'])

print(df_model[['leader_compound', 'leader_tyrelife', 'car_ahead_compound', 'car_ahead_tyrelife']].isnull().sum())

leader_compound       67744
leader_tyrelife       68595
car_ahead_compound    10262
car_ahead_tyrelife    17072
dtype: int64


**Teammate pit reaction**

In [72]:
teammate_pit_sum = df_model.groupby(['race_id', 'LapNumber', 'raw_team'])['is_pit_lap'].transform('sum')
own_pit = df_model['is_pit_lap'].astype(int)

df_model['teammate_pitted_this_lap'] = (teammate_pit_sum - own_pit) > 0

df_model['teammate_pitted_prev_lap'] = (
    df_model.sort_values(['race_id', 'raw_team', 'LapNumber'])
    .groupby(['race_id', 'raw_team'])['teammate_pitted_this_lap']
    .shift(1)
)
df_model['teammate_pitted_prev_lap'] = np.where(
    df_model['teammate_pitted_prev_lap'].isna(), False, df_model['teammate_pitted_prev_lap']
).astype(bool)

print(df_model['teammate_pitted_prev_lap'].value_counts())

teammate_pitted_prev_lap
False    241523
True      10815
Name: count, dtype: int64
